# Curating the CYP subset of the BindingDB IC50 deposition

`curate_data_IC50.ipynb` deposits BindingDB IC50 records permissively. This notebook reads that
deposition and applies the filtering needed before the CYP data can be used for model training.

The rules differ from the ChEMBL CYP curation, because BindingDB carries far less metadata: there is
no assay description, organism, assay id or patent id, so nothing here can detect time-dependent
inhibition, probe substrate, or a non-human orthologue. What BindingDB *does* have, and ChEMBL's
pChEMBL-filtered deposition does not, is censored (`>`, `<`) records.

What is removed:

- **censored** (`>`, `<`) — roughly half of all records. 
- **off-target** — `bindingdb_target_name_queried` is not the queried CYP
- **duplicate rows** — same compound, document and value
- **implausible** — an exact IC50 above 1 mM

What is flagged but kept, because it cannot be resolved from the data alone:

- **`unit_conflict`** — the compound carries two values a clean ~1000x apart, i.e. the same
  measurement curated once in nM and once in uM. No way to say which is right.
- **`constant_value_doc`** — every record in a document shares one value, which is a censoring
  threshold deposited without its `>` operator. 

## Parameters

In [4]:
CYP_TARGETS = {
    "CYP1A2": "P05177",
    "CYP2B6": "P20813",
    "CYP2C8": "P10632",
    "CYP2C9": "P11712",
    "CYP2C19": "P33261",
    "CYP2D6": "P10635",
    "CYP3A4": "P08684",
}

ENDPOINT_LABEL = 'IC50'
SOURCE = "BindingDB"
SOURCE_VERSION = '20261019'            # the deposition this curation reads
SOURCE_LOCATION = f"{SOURCE}{SOURCE_VERSION}_{ENDPOINT_LABEL}"
LOCATION = f"{SOURCE}{SOURCE_VERSION}_{ENDPOINT_LABEL}_curated"
BUCKET = "openadmet-data-public-dev"
PUSH_TO_S3 = False  # set True only after reviewing local output; never push unreviewed data

IMPLAUSIBLE_NM = 1e6     # an exact IC50 above 1 mM is not a measurement
UNIT_RATIO = 1000        # a "clean 1000x" pair means nM/uM confusion
UNIT_RTOL = 0.02
MIN_CONSTANT_ROWS = 4    # records a document needs before "all identical" means anything

In [5]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

source_path = Path(SOURCE_LOCATION)
location_path = Path(LOCATION)
location_path.mkdir(exist_ok=True)

## Curation rules

One predicate per reason a record is dropped.

In [6]:
# BindingDB affinities are nM with an optional leading censoring operator, and sometimes in
# scientific notation ("9.75e+3", ">5.0e+4").
AFFINITY = r"^\s*(?P<rel>[<>]=?|=)?\s*(?P<val>\d*\.?\d+(?:[eE][+-]?\d+)?)\s*$"

EXPECTED_NAME = {t: f"Cytochrome P450 {t[3:]}" for t in CYP_TARGETS}


def reparse_affinity(raw, target_name):
    """Re-derive relation and value from the original affinity string, and add pIC50."""
    parsed = raw["affinity"].str.extract(AFFINITY)
    raw = raw.copy()
    raw["standard_relation"] = parsed["rel"].fillna("=")
    raw["standard_value"] = parsed["val"].astype(float)
    raw["pIC50"] = 9 - np.log10(raw["standard_value"])
    raw["exact"] = raw["standard_relation"] == "="
    raw["off_target"] = raw["bindingdb_target_name_queried"] != EXPECTED_NAME[target_name]
    raw["duplicate"] = raw.duplicated(["OPENADMET_INCHIKEY", "doc_doi", "doc_pubmed_id",
                                       "standard_value", "standard_relation"])
    raw["implausible"] = raw["exact"] & (raw["standard_value"] > IMPLAUSIBLE_NM)
    if raw["standard_value"].isna().any():
        raise ValueError(f"{target_name}: {int(raw['standard_value'].isna().sum())} affinity strings "
                         f"could not be parsed, e.g. "
                         f"{raw.loc[raw['standard_value'].isna(), 'affinity'].unique()[:5].tolist()}")
    return raw


def removal_reasons(raw):
    """One boolean column per reason a raw record is dropped."""
    return pd.DataFrame({
        "off_target": raw["off_target"],
        "censored": ~raw["exact"],
        "implausible": raw["implausible"],
        "duplicate": raw["duplicate"],
    }, index=raw.index)


def unit_conflict_compounds(r, ratio=UNIT_RATIO, tol=UNIT_RTOL):
    """Compounds holding two values separated by (almost exactly) `ratio`."""
    hits = {}
    for k, g in r.groupby("OPENADMET_INCHIKEY"):
        v = np.sort(g["standard_value"].dropna().unique())
        if len(v) < 2:
            continue
        hi, lo = np.where(np.isclose(v[:, None] / v[None, :], ratio, rtol=tol))
        if len(hi):
            hits[k] = "; ".join(f"{v[j]:g} vs {v[i]:g}" for i, j in zip(lo, hi))
    return pd.Series(hits, dtype="object")


def constant_value_docs(r, min_rows=MIN_CONSTANT_ROWS):
    """Documents whose every record carries the same standard_value.
    """
    g = r.groupby("doc_doi")["standard_value"].agg(["size", "nunique"])
    return g.index[(g["size"] >= min_rows) & (g["nunique"] == 1)]

## Curate a target

In [7]:
META = ["OPENADMET_TARGET", "OPENADMET_TARGET_ID", "OPENADMET_ENDPOINT", "OPENADMET_UNITS",
        "OPENADMET_SOURCE", "OPENADMET_SOURCE_VERSION"]

CLEAN_COLS = ["bindingdb_target_name_queried", "bindingdb_monomer_id", "canonical_smiles",
              "standard_type", "affinity", "doc_pubmed_id", "doc_doi",
              "OPENADMET_CANONICAL_SMILES", "OPENADMET_INCHIKEY", "standard_relation",
              "standard_value", "pIC50", "unit_conflict", "constant_value_doc"] + META


def aggregate_by_compound(r):
    """Same grouping as the permissive deposition, plus pIC50 statistics and the flags."""
    data = r.groupby(["OPENADMET_CANONICAL_SMILES", "OPENADMET_INCHIKEY"]).agg(
        n_records=("standard_value", "size"),
        standard_value_mean=("standard_value", "mean"),
        standard_value_median=("standard_value", "median"),
        standard_value_std=("standard_value", "std"),
        pIC50_mean=("pIC50", "mean"),
        pIC50_median=("pIC50", "median"),
        pIC50_std=("pIC50", "std"),
        unit_conflict=("unit_conflict", "any"),
        constant_value_doc=("constant_value_doc", "any"),
    ).reset_index()
    for col in META:
        data[col] = r[col].iloc[0]
    return data.sort_values("n_records", ascending=False)


def curate_cyp_target(target_name: str, uniprot_id: str):
    print(f"working on target {target_name}")
    fname = f"{SOURCE}_{ENDPOINT_LABEL}_{target_name}_{uniprot_id}_raw.parquet"
    raw = reparse_affinity(pd.read_parquet(source_path / fname), target_name)
    print(f"  permissive records: {len(raw)}")

    # flagged, not dropped
    raw["unit_conflict"] = raw["OPENADMET_INCHIKEY"].isin(unit_conflict_compounds(raw).index)
    raw["constant_value_doc"] = raw["doc_doi"].isin(constant_value_docs(raw))

    reasons = removal_reasons(raw)
    keep = ~reasons.any(axis=1)
    clean = raw[keep][CLEAN_COLS].reset_index(drop=True)

    summary = {**reasons.sum().to_dict(),
               "raw_before": len(raw), "raw_after": int(keep.sum()),
               "compounds_after": clean["OPENADMET_INCHIKEY"].nunique(),
               "unit_conflict_flagged": int(clean["unit_conflict"].sum()),
               "constant_value_flagged": int(clean["constant_value_doc"].sum())}
    print(f"  dropped {int((~keep).sum())} "
          f"({', '.join(f'{k} {int(v)}' for k, v in reasons.sum().items() if v)})")
    print(f"  kept {len(clean)} records over {summary['compounds_after']} compounds")
    return clean, aggregate_by_compound(clean), summary

## Main loop

Curate each target and save to parquet, in the same layout as the permissive deposition.

In [8]:
summaries = {}
for target, uniprot_id in CYP_TARGETS.items():
    clean, agg, summary = curate_cyp_target(target, uniprot_id)
    summaries[target] = summary

    fname_agg = f"{SOURCE}_{ENDPOINT_LABEL}_CYP_{target}_{uniprot_id}_aggregated.parquet"
    fname_raw = f"{SOURCE}_{ENDPOINT_LABEL}_CYP_{target}_{uniprot_id}_raw.parquet"
    agg.reset_index(drop=True).to_parquet(location_path / fname_agg, index=False)
    clean.reset_index(drop=True).to_parquet(location_path / fname_raw, index=False)
    print(f"  wrote {fname_raw} (rows={len(clean)}), {fname_agg} (rows={len(agg)})")

working on target CYP1A2
  permissive records: 4275
  dropped 2749 (off_target 2, censored 2669, implausible 10, duplicate 134)
  kept 1526 records over 1382 compounds
  wrote BindingDB_IC50_CYP_CYP1A2_P05177_raw.parquet (rows=1526), BindingDB_IC50_CYP_CYP1A2_P05177_aggregated.parquet (rows=1382)
working on target CYP2B6
  permissive records: 546
  dropped 302 (off_target 1, censored 301, duplicate 19)
  kept 244 records over 213 compounds
  wrote BindingDB_IC50_CYP_CYP2B6_P20813_raw.parquet (rows=244), BindingDB_IC50_CYP_CYP2B6_P20813_aggregated.parquet (rows=213)
working on target CYP2C8
  permissive records: 984
  dropped 516 (off_target 1, censored 508, implausible 1, duplicate 25)
  kept 468 records over 435 compounds
  wrote BindingDB_IC50_CYP_CYP2C8_P10632_raw.parquet (rows=468), BindingDB_IC50_CYP_CYP2C8_P10632_aggregated.parquet (rows=435)
working on target CYP2C9
  permissive records: 5750
  dropped 2983 (off_target 1, censored 2935, implausible 1, duplicate 120)
  kept 2767 

In [9]:
summary_table = pd.DataFrame(summaries).T
summary_table["% kept"] = (100 * summary_table["raw_after"] / summary_table["raw_before"]).round(1)
summary_table.to_csv(location_path / f"curation_summary_{LOCATION}.csv")
summary_table

,off_target,censored,implausible,duplicate,raw_before,raw_after,compounds_after,unit_conflict_flagged,constant_value_flagged,% kept
CYP1A2,2.0,2669.0,10.0,134.0,4275.0,1526.0,1382.0,47.0,12.0,35.7
CYP2B6,1.0,301.0,0.0,19.0,546.0,244.0,213.0,0.0,1.0,44.7
CYP2C8,1.0,508.0,1.0,25.0,984.0,468.0,435.0,2.0,0.0,47.6
CYP2C9,1.0,2935.0,1.0,120.0,5750.0,2767.0,2548.0,53.0,12.0,48.1
CYP2C19,11.0,2017.0,2.0,48.0,3549.0,1510.0,1393.0,4.0,10.0,42.5
CYP2D6,0.0,3619.0,9.0,173.0,6674.0,2986.0,2721.0,59.0,12.0,44.7
CYP3A4,1.0,4734.0,11.0,689.0,10550.0,5619.0,4739.0,28.0,28.0,53.3


## Push to S3 and build the Intake catalog

Guarded by `PUSH_TO_S3`, and written under this deposition's own `LOCATION`, so the permissive
catalog is untouched.

In [ ]:
if PUSH_TO_S3:
    import intake
    from intake.readers.datatypes import Parquet

    from openadmet.toolkit.webservices.credentials import S3Settings
    from openadmet.toolkit.webservices.s3 import S3Bucket

    assert "AWS_ACCESS_KEY_ID" in __import__("os").environ, "set AWS credentials in the environment first"
    settings = S3Settings()
    bucket = S3Bucket.from_settings(settings, BUCKET)

    cat = intake.entry.Catalog()
    for target, uniprot_id in CYP_TARGETS.items():
        fname_agg = f"{SOURCE}_{ENDPOINT_LABEL}_CYP_{target}_{uniprot_id}_aggregated.parquet"
        fname_raw = f"{SOURCE}_{ENDPOINT_LABEL}_CYP_{target}_{uniprot_id}_raw.parquet"
        agg_path, raw_path = location_path / fname_agg, location_path / fname_raw
        if not (agg_path.exists() and raw_path.exists()):
            continue
        for path, fname, subset in [(agg_path, fname_agg, "aggregated"), (raw_path, fname_raw, "raw")]:
            dest = f"{LOCATION}/{fname}"
            bucket.push_file(path, dest)
            uri = bucket.to_uri(dest)
            row_desc = "one row per compound" if subset == "aggregated" else "one row per source record"
            metadata = {
                "description": (
                    f"{SOURCE}{SOURCE_VERSION} {ENDPOINT_LABEL} activity data for {target} "
                    f"({uniprot_id}), CYP-curated (exact values only), {row_desc}."
                ),
                "units": "nM",
                "endpoint": ENDPOINT_LABEL,
                "target": target,
                "target_uniprot_id": uniprot_id,
                "source_version": SOURCE_VERSION,
                "curation": "CYP subset: censored, off-target, duplicate and implausible records removed",
            }
            cat[f"{target}_{subset}"] = intake.readers.PandasParquet(Parquet(url=uri, metadata=metadata))

    catname = f"CATALOG_{LOCATION}.yaml"
    cat.to_yaml_file(catname)
    bucket.push_file(catname, f"{LOCATION}/{catname}")
    print(f"pushed catalog to {LOCATION}/{catname}")
else:
    print("PUSH_TO_S3 is False -- local parquet only, nothing pushed, no catalog built")